# 🎬 Final Project — Multimodal AI ChatBot for YouTube Video Q&A
### Use Case: Science Documentaries (Balade Mentale)

**Complete Workflow — RAG Pipeline + LangGraph Agent**

This notebook builds a full Retrieval-Augmented Generation (RAG) system that:
- Extracts transcripts from YouTube videos
- Chunks and embeds them into a persistent ChromaDB vector store
- Answers questions in French or English using a local LLM (Ollama)
- Uses a LangGraph agent with memory and 3 custom tools
- Is served via a Streamlit web interface (`app.py`)

**Stack:** LangChain · ChromaDB · HuggingFace Embeddings · Ollama (llama3.1:8b) · LangSmith · Streamlit

## 📦 STEP 0 — Install Dependencies

Run this cell **once only**, then restart the kernel.

In [ ]:
%pip install -q langchain langchain-community langchain-core
%pip install -q langchain-huggingface langchain-chroma langchain-ollama
%pip install -q chromadb youtube-transcript-api yt-dlp
%pip install -q sentence-transformers transformers accelerate bitsandbytes
%pip install -q langsmith python-dotenv tqdm
%pip install -q langchain-text-splitters langdetect langgraph
%pip install -q sounddevice scipy openai-whisper

print("✅ All libraries installed!")

## 🔑 STEP 1 — LangSmith Configuration

LangSmith provides observability for our RAG pipeline: every call to the LLM, 
retriever, and tools is traced and visible in the LangSmith dashboard.

Add your `LANGCHAIN_API_KEY` to a `.env` file at the project root.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"]    = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"]    = "youtube-rag-balade-mentale"  # LangSmith project

if os.environ.get("LANGCHAIN_API_KEY"):
    print("✅ LangSmith configured — traces visible at https://smith.langchain.com")
else:
    print("⚠️  LANGCHAIN_API_KEY missing! Create a .env file with your key.")

## 📥 STEP 2 — YouTube Transcript Extraction

We use `youtube-transcript-api` to fetch the auto-generated captions from a YouTube video. 
The transcript is returned as a list of timed segments which we concatenate into a single text string.

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
import re

VIDEO_URL = "https://youtu.be/300eyzsqang?si=YMrPUffy3kqjz6bK"
VIDEO_ID  = re.search(r"(?:v=|youtu\.be/)([\w-]{11})", VIDEO_URL).group(1)

print(f"🎯 Video ID extracted: {VIDEO_ID}")

def extract_transcript_api(video_id: str) -> str | None:
    try:
        ytt = YouTubeTranscriptApi()
        transcript = ytt.fetch(video_id, languages=['fr', 'en'])
        full_text = " ".join([seg.text.strip() for seg in transcript])
        print(f"✅ Transcript extracted: {len(full_text)} characters")
        return full_text
    except Exception as e:
        print(f"⚠️  No transcript available: {e}")
        return None

raw_transcript = extract_transcript_api(VIDEO_ID)

if raw_transcript:
    print(f"\n📝 Preview of first 500 characters:")
    print("-" * 60)
    print(raw_transcript[:500])
else:
    print("\n❌ Unable to extract transcript.")

## 🧹 STEP 3 — Transcript Cleaning

Raw transcripts contain HTML entities, tags, and extra whitespace. 
This step normalises the text before chunking.

In [ ]:
import html

def clean_transcript(text: str) -> str:
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\[[^\]]*\]', '', text)
    text = re.sub(r'[\t\r\f\v]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    text = text.strip()
    return text

cleaned_transcript = clean_transcript(raw_transcript)

print(f"📊 Before cleaning : {len(raw_transcript):,} characters")
print(f"📊 After cleaning  : {len(cleaned_transcript):,} characters")

## ✂️ STEP 4 — Chunking with LangChain

We split the transcript into overlapping chunks using `RecursiveCharacterTextSplitter`.

**Parameters chosen:**
- `chunk_size=800` — fits comfortably in the LLM context window
- `chunk_overlap=150` — preserves continuity between adjacent chunks
- `separators` — respects sentence boundaries to avoid splitting mid-sentence

Each chunk is wrapped in a LangChain `Document` with metadata: `video_id`, `source URL`, `channel`, `topic`, `language`.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " ", ""]
)

doc = Document(
    page_content=cleaned_transcript,
    metadata={
        "source": VIDEO_URL,
        "video_id": VIDEO_ID,
        "channel": "Balade Mentale",
        "topic": "marine_biology",
        "language": "fr",
    }
)

chunks = text_splitter.split_documents([doc])

chunk_sizes = [len(c.page_content) for c in chunks]
print(f"📦 Number of chunks created  : {len(chunks)}")
print(f"📏 Min / max / avg size      : {min(chunk_sizes)} / {max(chunk_sizes)} / {sum(chunk_sizes)//len(chunk_sizes)} characters")

## 🧮 STEP 5 — Embeddings + ChromaDB Vector Store

We use `intfloat/multilingual-e5-large` — a state-of-the-art multilingual embedding model 
that handles both French and English queries natively. Embeddings are computed on GPU (CUDA) for speed.

ChromaDB persists the vector store locally in `./chroma_db`, so we don't need to re-embed on every run.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

print("\n⏳ Loading embeddings model...")
embeddings_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32}
)
print("✅ Embeddings model loaded!")

CHROMA_DIR = "./chroma_db"
COLLECTION = "balade_mentale_biologie_marine"

print(f"\n💾 Storing in ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings_model,
    persist_directory=CHROMA_DIR,
    collection_name=COLLECTION
)
print(f"✅ {len(chunks)} chunks vectorized and stored!")

## 🔍 STEP 6 — Vector Search Test

We validate the vector store by running a few test queries in both French and English. 
The retriever should return relevant chunks even when the query language differs from the transcript language, 
thanks to the multilingual embedding model.

In [ ]:
vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings_model,
    collection_name=COLLECTION
)
count = vectorstore._collection.count()
print(f"Chunks: {count}")

questions = [
    "Comment les pieuvres utilisent-elles leurs ventouses ?",
    "What is the vampire squid ?",
    "Pourquoi les créatures des abysses sont-elles transparentes ?",
]

for test_query in questions:
    print(f"\n🔍 Query: '{test_query}'")
    print("=" * 70)
    results = vectorstore.similarity_search_with_score(query=test_query, k=3)
    for i, (doc, score) in enumerate(results, 1):
        print(f"\n📄 Chunk #{i} — Score : {score:.4f}")
        print("-" * 50)
        print(doc.page_content[:300])
        print(f"Source : {doc.metadata.get('source', 'N/A')}")

best_score = results[0][1] if results else 1
if best_score < 0.3:
    print("\n🟢 Excellent !")
elif best_score < 0.5:
    print("\n🟡 Correct.")
else:
    print("\n🔴 Score faible.")

## 🎯 STEP 6b — Retriever Configuration

We use **MMR (Maximal Marginal Relevance)** as the search strategy instead of plain similarity search.

MMR balances relevance and diversity: it avoids returning 4 near-identical chunks from the same passage, 
and instead selects chunks that cover different aspects of the answer. This gives the LLM richer context.

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 20,
        "lambda_mult": 0.7
    }
)

print("✅ Retriever defined !")
print(f"   → MMR deduplication, returns {retriever.search_kwargs['k']} chunks per question")

## 🤖 STEP 7 — Local LLM via Ollama

We use `llama3.1:8b` running locally via Ollama — no API key, no latency, full privacy.

**Key settings:**
- `temperature=0.1` — near-deterministic answers, better for factual Q&A
- `num_ctx=8192` — large context window to hold conversation history + retrieved chunks

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0.1,
    num_ctx=8192,
)

try:
    response = llm.invoke([HumanMessage(content="Answer in one sentence: what is an LLM?")])
    print("✅ llama3.1:8b is working!")
    print(f"Response: {response.content}")
except Exception as e:
    print(f"⚠️  Error: {e}")

## ⛓️ STEP 8 — Bilingual RAG Pipeline

The RAG chain connects the retriever, prompt, LLM, and output parser into a single runnable pipeline.

We use **two separate prompts** (French and English) and detect the query language automatically. 
The French prompt tells the LLM to answer only from the transcript context; 
the English prompt additionally enforces an English response.

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langdetect import detect

RAG_PROMPT_FR = PromptTemplate(
    input_variables=["context", "question"],
    template="""Tu es un assistant expert en sciences naturelles spécialisé dans les vidéos Balade Mentale.
Réponds aux questions en te basant UNIQUEMENT sur les extraits de transcript fournis.
Si la réponse n'est pas dans le contexte, dis clairement "Cette information n'est pas dans la vidéo."
Réponds de façon claire et pédagogique.

CONTEXTE :
{context}

QUESTION : {question}

RÉPONSE :"""
)

RAG_PROMPT_EN = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an expert assistant specialized in Balade Mentale science videos.
Answer questions based ONLY on the transcript excerpts provided.
If the answer is not in the context, clearly say "This information is not in the video."
You MUST respond in English.

CONTEXT:
{context}

QUESTION: {question}

ANSWER IN ENGLISH:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def detect_language(text):
    try:
        return detect(text)
    except:
        return "en"

def ask(question):
    lang = detect_language(question)
    prompt = RAG_PROMPT_FR if lang == "fr" else RAG_PROMPT_EN
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )
    return chain.invoke(question)

print("✅ Chaîne RAG bilingue prête !")

# Tests rapides
for q in ["Qu'est-ce que la bioluminescence ?", "What are deep sea creatures ?"]:
    print(f"\nQ: {q}")
    print(f"A: {ask(q)[:200]}...")

## 📹 STEP 9 — Multi-Video Ingestion

> ⚠️ **Do not re-run this cell** — all 280 chunks are already persisted in ChromaDB.

This step ingests all 7 Balade Mentale videos. For each video it:
1. Extracts the transcript via `youtube-transcript-api`
2. Cleans the text
3. Splits it into chunks with metadata (`video_id`, `topic`, `channel`)
4. Adds the chunks to ChromaDB, skipping any already-indexed videos

**Result:** 280 chunks across 7 videos stored in `./chroma_db`

In [ ]:
# ⚠️ DO NOT RE-RUN — already ingested in ChromaDB (280 chunks)
# YouTube blocks repeated requests

VIDEOS = [
    "https://youtu.be/BXebijm_Mao?si=bZRIR3vDZxepwTQq",
    "https://youtu.be/2KQYrXJ_0N4?si=StXmChnKeNYvP_Qn",
    "https://youtu.be/NDeyeTQ-ZAM?si=yZZSgBEVz1iXkV0X",
    "https://youtu.be/2--n3nStqTk?si=JCyD5CpdxMf6tKtm",
    "https://youtu.be/86-T834tOU8?si=ZSWOOfCuTz1e61tG",
    "https://youtu.be/NFJk2fmIidU?si=6S5lvJb8O3J0y6pT",
]

def extract_video_id(url):
    return re.search(r"(?:v=|youtu\.be/)([\w-]{11})", url).group(1)

def process_video(url):
    video_id = extract_video_id(url)
    print(f"\n🎬 Processing: {video_id}")
    try:
        ytt = YouTubeTranscriptApi()
        transcript = ytt.fetch(video_id, languages=['fr', 'en'])
        full_text = " ".join([seg.text.strip() for seg in transcript])
        cleaned = clean_transcript(full_text)
        doc = Document(
            page_content=cleaned,
            metadata={"source": url, "video_id": video_id, "channel": "Balade Mentale", "topic": "sciences", "language": "fr"}
        )
        new_chunks = text_splitter.split_documents([doc])
        vectorstore.add_documents(new_chunks)
        print(f"   ✅ {len(new_chunks)} chunks added")
        return len(new_chunks)
    except Exception as e:
        print(f"   ⚠️ Error: {e}")
        return 0

# Check how many chunks are already in ChromaDB
count = vectorstore._collection.count()
print(f"📊 Chunks currently in ChromaDB: {count}")
print("✅ If count >= 280, no need to re-run ingestion!")

In [ ]:
# Verify total chunks in ChromaDB
count = vectorstore._collection.count()
print(f"✅ Total chunks in ChromaDB: {count}")

results = vectorstore._collection.get()
video_ids = set(m['video_id'] for m in results['metadatas'])
print(f"📹 Videos indexed: {len(video_ids)}")
for vid in sorted(video_ids):
    print(f"   · {vid}")

## 🛠️ STEP 10 — LangChain Tools (3 tools)

We define 3 tools that the LangGraph agent can call:

1. **`rag_search`** — searches ChromaDB for relevant transcript chunks and generates an answer
2. **`add_youtube_video`** — ingests a new YouTube video into the vector store at runtime
3. **`list_available_topics`** — lists all 7 indexed videos with their topics

Each tool has a docstring that describes when and how to use it — the agent reads these to decide which tool to call.

In [ ]:
from langchain.tools import tool

@tool
def rag_search(question: str) -> str:
    """Search the ChromaDB knowledge base of Balade Mentale videos.
    Use this tool to answer questions about the video content.
    Supports both French and English questions."""
    if isinstance(question, dict):
        question = question.get("value") or question.get("description") or str(question)
    lang = detect_language(question)
    prompt = RAG_PROMPT_FR if lang == "fr" else RAG_PROMPT_EN
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )
    return chain.invoke(question)

@tool
def add_youtube_video(url: str) -> str:
    """Extract the transcript of a new YouTube video and add it to ChromaDB.
    Use this tool when the user wants to add a new video to the knowledge base."""
    if isinstance(url, dict):
        url = url.get("value") or url.get("description") or str(url)
    try:
        total = process_video(url)
        return f"✅ Vidéo ajoutée ! {total} chunks dans ChromaDB."
    except Exception as e:
        return f"⚠️ Erreur : {e}"

@tool
def list_available_topics() -> str:
    """List all available videos and topics in the knowledge base.
    Use this tool when the user asks what topics are available, what videos are in the database,
    or what they can ask questions about. Works in French and English."""
    videos = [
        ("Les pieuvres vampires des Abysses, survivantes de l'impossible",
         "Vampire Squid from the Abyss, survivors of the impossible",
         "biologie marine, céphalopodes, bioluminescence / marine biology, cephalopods, deep sea"),
        ("Des milliards d'autres planètes !",
         "Billions of other planets!",
         "astronomie, exoplanètes / astronomy, exoplanets"),
        ("LE GIGANTISME des Abysses",
         "The GIGANTISM of the Abyss",
         "biologie marine, gigantisme / marine biology, gigantism"),
        ("On a filmé des créatures géantes et transparentes dans les abysses",
         "We filmed giant transparent creatures in the abyss",
         "biologie marine, créatures géantes / marine biology, giant creatures"),
        ("La folle Histoire de notre système solaire",
         "The crazy History of our solar system",
         "astronomie, système solaire / astronomy, solar system"),
        ("Ces lacs sous-marins qui ne devraient pas exister",
         "These underwater lakes that shouldn't exist",
         "océanographie, lacs sous-marins / oceanography, underwater lakes"),
        ("Le froid maximal - Voyage vers le zéro absolu",
         "Maximum cold - Journey to absolute zero",
         "physique, zéro absolu / physics, absolute zero"),
    ]
    result = "📚 Available videos / Vidéos disponibles :\n\n"
    for fr, en, topics in videos:
        result += f"• 🇫🇷 {fr}\n  🇬🇧 {en}\n  Topics : {topics}\n\n"
    result += "Posez vos questions en FR ou EN ! / Ask me anything in French or English!"
    return result

tools = [rag_search, add_youtube_video, list_available_topics]
print("✅ 3 tools disponibles !")
print(f"   → {[t.name for t in tools]}")

## 🤖 STEP 11 — LangGraph Agent + Conversational Memory

We use LangGraph's `create_react_agent` to build a ReAct agent (Reason + Act). 
The agent iteratively reasons about which tool to call, calls it, observes the result, and refines its answer.

**Memory** is implemented via a `chat_history` list of `HumanMessage` / `AIMessage` objects. 
The last 6 exchanges are injected into every prompt so the agent can answer follow-up questions correctly.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import AIMessage, HumanMessage

SYSTEM_PROMPT = """You are a helpful assistant specialized in Balade Mentale YouTube videos about science.
Answer in the same language as the user (French if French, English if English).

BEHAVIOR RULES:

1. FOR SCIENCE QUESTIONS:
   - ALWAYS use rag_search tool first to search in the video knowledge base.
   - If rag_search returns a good answer → give it to the user clearly.
   - If rag_search returns "Cette information n'est pas dans la vidéo" or "This information is not in the video" → tell the user:
     "Cette information n'est pas disponible dans les vidéos Balade Mentale. Souhaitez-vous que je réponde avec mes connaissances générales ?"
     Then WAIT for the user to confirm.
   - If the user confirms → answer but ALWAYS add:
     "⚠️ Cette réponse provient de mes connaissances générales et non des vidéos Balade Mentale."

2. FOR TOPIC/VIDEO QUESTIONS:
   - ALWAYS use list_available_topics tool.

3. FOR ADDING A VIDEO:
   - ALWAYS use add_youtube_video tool when the user provides a YouTube URL.

4. FOR GREETINGS & CASUAL CHAT:
   - Answer directly and naturally without using any tool.

5. NEVER invent information about the videos.
6. NEVER answer science questions from your own knowledge WITHOUT warning the user first.
7. Give ONLY the final answer — never explain that you are using a tool."""

agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

print("✅ LangGraph Agent ready!")

# ── MEMORY ───────────────────────────────────────────────────────────────────
chat_history = []

def chat(question):
    global chat_history

    # Direct detection of topic-related questions
    topic_keywords = ["sujets", "topics", "disponibles", "vidéos", "videos",
                      "available", "what can", "quoi poser", "de quoi"]
    if any(kw in question.lower() for kw in topic_keywords):
        answer = list_available_topics.invoke({})
        chat_history.append(HumanMessage(content=question))
        chat_history.append(AIMessage(content=answer))
        return answer

    # Greeting detection → skip RAG
    casual_keywords = ["bonjour", "hello", "salut", "hi", "merci", "thanks",
                       "au revoir", "bye", "comment tu t'appelles", "qui es-tu"]
    if any(kw in question.lower() for kw in casual_keywords):
        chat_history.append(HumanMessage(content=question))
        response = agent_executor.invoke({"messages": chat_history})
        answer = ""
        for msg in reversed(response["messages"]):
            if msg.__class__.__name__ == "AIMessage" and msg.content:
                content = msg.content.strip()
                if not content.startswith("{") and not content.startswith("[{"):
                    answer = content
                    break
        # Fallback if response is empty
        if not answer:
            answer = "Hello! I'm your Balade Mentale assistant. Ask me anything about the science videos! 🎬"
        chat_history.append(AIMessage(content=answer))
        return answer

    # For EVERYTHING else → rag_search directly
    chat_history.append(HumanMessage(content=question))
    answer = rag_search.invoke(question)
    chat_history.append(AIMessage(content=answer))
    return answer

# ── TESTS ────────────────────────────────────────────────────────────────────
print("\n🧪 Test 1 — Greeting...")
print(chat("Hello! What's your name?"))

print("\n🧪 Test 2 — Science question...")
print(chat("What is bioluminescence?"))

print("\n🧪 Test 3 — Memory...")
print(chat("And how do these creatures defend themselves?"))

print("\n🧪 Test 4 — Available topics...")
print(chat("What topics are available?"))

## 🎤 STEP 12 — Speech Recognition with Whisper *(Attempted — Optional)*

> ⚠️ **Status: Not functional in this project** — `ffmpeg` is required but could not be installed on Windows due to a missing DLL error (`libssl-1_1-x64.dll`). The code below was written and tested conceptually but could not be executed.

### What was attempted

The goal was to add voice input to the chatbot using OpenAI's Whisper model running locally on GPU. 
The user would click a button in the Streamlit interface, speak their question, and Whisper would transcribe it before passing it to the RAG pipeline — eliminating the need to type.

### Why it didn't work

`whisper` depends on `ffmpeg` to decode audio files. On Windows, installing `ffmpeg` via `conda` triggered a DLL conflict that broke the environment. 
A clean Linux or macOS environment would not have this issue.

### How to make it work

On Linux/macOS: `conda install ffmpeg -c conda-forge`, then `pip install openai-whisper sounddevice scipy`. 
The code below would then run without modification.

In [ ]:
# ── STATUS: Not executed — ffmpeg DLL error on Windows ──────────────────────
# This code is complete and correct but requires ffmpeg to be installed.
# To enable: conda install ffmpeg -c conda-forge (Linux/macOS)
# ─────────────────────────────────────────────────────────────────────────────

import whisper
import sounddevice as sd
import numpy as np
from scipy.io.wavfile import write
import tempfile, os, torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⏳ Loading Whisper sur {device}...")
whisper_model = whisper.load_model("base", device=device)
print("✅ Whisper chargé !")

def record_audio(duration=5, sample_rate=16000):
    print(f"🎤 Enregistrement {duration}s... Parlez !")
    audio = sd.rec(
        int(duration * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype=np.float32,
        device=4  # Razer Seiren Mini
    )
    sd.wait()
    print("⏹️  Terminé !")
    return audio, sample_rate

def transcribe_audio(audio, sample_rate):
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        temp_path = f.name
        write(temp_path, sample_rate, audio)
    result = whisper_model.transcribe(temp_path, language=None)
    os.unlink(temp_path)
    text = result["text"].strip()
    print(f"📝 Transcrit ({result['language']}) : {text}")
    return text

def voice_chat(duration=5):
    """Enregistre, transcrit et envoie à l'agent"""
    audio, sr = record_audio(duration)
    question = transcribe_audio(audio, sr)
    if not question:
        return "⚠️ Aucun audio détecté."
    print(f"\n🤔 Question : {question}")
    response = chat(question)
    print(f"\n💬 RÉPONSE : {response}")
    return response

print("\n✅ Whisper ready ! Lance voice_chat(duration=5) pour tester.")
print("   → Nécessite ffmpeg : conda install ffmpeg -c conda-forge")

## 📊 STEP 13 — RAG Evaluation with LangSmith

Now that the full pipeline is working, we evaluate it systematically using LangSmith's evaluation framework.

### How it works

We define a **ground-truth dataset** of question/answer pairs that we know to be correct. 
LangSmith then runs our RAG pipeline on each question and uses an LLM-as-judge to compare 
the generated answer against the expected answer. This gives us objective, reproducible metrics.

### Metrics evaluated

**Correctness** — Does the generated answer convey the same meaning as the expected answer? 
This is scored 0–1 by a judge LLM that reads both answers and rates their semantic equivalence.

**Faithfulness** — Does the answer stick to what was in the retrieved chunks, or does the LLM hallucinate? 
A low faithfulness score means the LLM is going beyond the transcript context.

### Dataset design

We include 3 types of questions: questions well covered by the videos (expected high score), 
a bilingual question to test the FR/EN pipeline, and an out-of-scope question to verify the fallback behaviour.

In [2]:
from langsmith import Client
from langsmith.evaluation import evaluate
from langchain_core.output_parsers import StrOutputParser

# ── 1. LangSmith client ───────────────────────────────────────────────────────
ls_client = Client()

# ── 2. Ground-truth evaluation dataset ───────────────────────────────────────
EVAL_EXAMPLES = [
    {
        "input": "Qu'est-ce que la bioluminescence et comment les créatures des abysses l'utilisent-elles ?",
        "output": "La bioluminescence est la capacité de certains organismes à produire de la lumière "
                  "chimiquement. Les créatures des abysses l'utilisent pour attirer des proies, "
                  "communiquer ou repousser des prédateurs."
    },
    {
        "input": "Comment s'est formé le système solaire ?",
        "output": "Le système solaire s'est formé il y a environ 4,6 milliards d'années à partir "
                  "d'un nuage de gaz et de poussières. Le Soleil s'est formé au centre, "
                  "et les planètes se sont agrégées dans le disque protoplanétaire."
    },
    {
        "input": "What is absolute zero and why can't we reach it?",
        "output": "Absolute zero is 0 Kelvin or -273.15°C, the temperature at which all thermal "
                  "motion theoretically stops. It cannot be reached due to the third law of thermodynamics."
    },
    {
        "input": "Pourquoi les lacs sous-marins ne se mélangent-ils pas à l'eau de mer ?",
        "output": "Les lacs sous-marins sont composés de saumure très dense qui ne se mélange pas "
                  "à l'eau de mer environnante moins dense, créant une interface stable."
    },
    {
        "input": "Qu'est-ce que le gigantisme des abysses ?",
        "output": "Le gigantisme des abysses est le phénomène par lequel certaines espèces marines "
                  "profondes atteignent des tailles bien supérieures à leurs équivalents de surface, "
                  "lié à la pression, la température et la rareté des ressources."
    },
    {
        "input": "Quelles créatures des abysses sont transparentes et pourquoi ?",
        "output": "Des créatures comme les méduses et certains calmars sont transparentes pour "
                  "se camoufler dans l'obscurité des abysses et échapper aux prédateurs."
    },
    {
        "input": "How many exoplanets have been discovered?",
        "output": "Thousands of exoplanets have been confirmed, with estimates suggesting "
                  "billions of planets exist in our galaxy."
    },
    {
        "input": "Quel est le taux de chômage en France ?",
        "output": "Cette question est hors du domaine des vidéos Balade Mentale. "
                  "Le système devrait indiquer que l'information n'est pas dans les vidéos."
    },
]

# ── 3. Create dataset in LangSmith ───────────────────────────────────────────
DATASET_NAME = "balade-mentale-rag-eval-v1"

# Clean up existing dataset for reproducibility
existing = [d for d in ls_client.list_datasets() if d.name == DATASET_NAME]
if existing:
    ls_client.delete_dataset(dataset_id=existing[0].id)
    print(f"🗑️  Deleted existing dataset: {DATASET_NAME}")

dataset = ls_client.create_dataset(
    dataset_name=DATASET_NAME,
    description="Ground-truth Q&A pairs for Balade Mentale RAG evaluation"
)
ls_client.create_examples(
    inputs=[{"question": e["input"]} for e in EVAL_EXAMPLES],
    outputs=[{"answer": e["output"]} for e in EVAL_EXAMPLES],
    dataset_id=dataset.id
)
print(f"✅ Dataset created: {DATASET_NAME} ({len(EVAL_EXAMPLES)} examples)")


c:\Users\rache\anaconda3\envs\youtube-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Dataset created: balade-mentale-rag-eval-v1 (8 examples)


In [7]:
from langsmith.evaluation import evaluate
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import torch

# ── Constantes définies ici pour être accessibles dans les closures ───────────
_CHROMA_DIR  = "./chroma_db"
_COLLECTION  = "balade_mentale_sciences"
_EMBED_MODEL = "intfloat/multilingual-e5-large"
_LLM_MODEL   = "llama3.1:8b"

# ── On instancie tout ici, dans ce scope de cellule ───────────────────────────
# LangSmith exécute les fonctions dans des threads séparés.
# La seule façon fiable est de capturer les objets dans une closure
# en les passant comme arguments par défaut (default argument trick).

_device = "cuda" if torch.cuda.is_available() else "cpu"

_embeddings = HuggingFaceEmbeddings(
    model_name=_EMBED_MODEL,
    model_kwargs={"device": _device},
    encode_kwargs={"normalize_embeddings": True}
)
_vectorstore = Chroma(
    persist_directory=_CHROMA_DIR,
    embedding_function=_embeddings,
    collection_name=_COLLECTION
)
_retriever = _vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.7}
)
_llm = ChatOllama(model=_LLM_MODEL, temperature=0.1, num_ctx=8192)

_prompt_fr = PromptTemplate(
    input_variables=["context", "question"],
    template="""Tu es un assistant expert en vidéos Balade Mentale.
Réponds UNIQUEMENT depuis le contexte. Si absent, dis "Cette information n'est pas dans la vidéo."
CONTEXTE : {context}
QUESTION : {question}
RÉPONSE :"""
)
_prompt_en = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are an expert on Balade Mentale videos.
Answer ONLY from the context. If absent, say "This information is not in the video."
CONTEXT: {context}
QUESTION: {question}
ANSWER:"""
)

# ── Pipeline wrapper — tout capturé en closure via arguments par défaut ────────
def rag_pipeline_for_eval(
    inputs,
    retriever=_retriever,
    llm=_llm,
    prompt_fr=_prompt_fr,
    prompt_en=_prompt_en
):
    question = inputs["question"]
    lang = "fr" if any(w in question.lower()
                       for w in ["qu", "est", "comment", "pourquoi", "quelles", "quel"]) else "en"
    prompt = prompt_fr if lang == "fr" else prompt_en

    docs    = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    chain   = ({"context": lambda _: context, "question": RunnablePassthrough()}
               | prompt | llm | StrOutputParser())
    answer  = chain.invoke(question)

    not_found = "pas dans la vidéo" in answer.lower() or "not in the video" in answer.lower()
    if not_found:
        fallback = llm.invoke(f"Answer clearly and educationally: {question}").content
        answer   = f"⚠️ Not in videos — general answer:\n\n{fallback}"

    return {"answer": answer}


# ── Evaluator — llm capturé de la même façon ──────────────────────────────────
def correctness_evaluator(run, example, llm=_llm):
    generated = run.outputs.get("answer", "")
    expected  = example.outputs.get("answer", "")
    question  = example.inputs.get("question", "")

    verdict = llm.invoke(
        f"Question: {question}\n"
        f"Expected: {expected}\n"
        f"Generated: {generated}\n"
        f"Is the generated answer correct? Reply ONLY: CORRECT or INCORRECT."
    ).content.strip().upper()

    return {"key": "correctness", "score": 1 if "CORRECT" in verdict else 0}


# ── Run ───────────────────────────────────────────────────────────────────────
print("⏳ Running evaluation on 8 questions...")

results = evaluate(
    rag_pipeline_for_eval,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix="balade-mentale-rag",
    metadata={"model": _LLM_MODEL, "embeddings": _EMBED_MODEL, "chunks": 280}
)

print("\n✅ Done! View results at https://smith.langchain.com")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4970.58it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⏳ Running evaluation on 8 questions...
View the evaluation results for experiment: 'balade-mentale-rag-b38f249a' at:
https://smith.langchain.com/o/8a6e7381-5540-4e3c-9cda-14b831ed2214/datasets/61a2ebb5-d4d4-499c-a2e6-6f6604a9aaca/compare?selectedSessions=dcdaac0c-48fa-44f8-9d7f-5bcf1e5296e6




8it [00:27,  3.47s/it]


✅ Done! View results at https://smith.langchain.com


In [4]:
# ── 7. Display results summary locally ───────────────────────────────────────
import pandas as pd

rows = []
for r in results._results:
    question  = r["example"].inputs["question"]
    expected  = r["example"].outputs["answer"]
    generated = r["run"].outputs.get("answer", "ERROR")
    score     = None
    for fb in r["evaluation_results"]["results"]:
        if hasattr(fb, "score"):
            score = fb.score
            break
    rows.append({
        "Question":          question[:60] + "..." if len(question) > 60 else question,
        "Expected (short)":  expected[:80] + "..." if len(expected) > 80 else expected,
        "Generated (short)": generated[:80] + "..." if len(generated) > 80 else generated,
        "Score":             score,
    })

df = pd.DataFrame(rows)
avg_score = df["Score"].dropna().mean()

print(f"📊 Evaluation Summary")
print(f"   Questions evaluated : {len(df)}")
print(f"   Average score       : {avg_score:.2f} / 1.00")
print(f"   Questions scored 1  : {(df['Score'] == 1).sum()}")
print(f"   Questions scored 0  : {(df['Score'] == 0).sum()}")
print()
print(df.to_string(index=False))


📊 Evaluation Summary
   Questions evaluated : 8
   Average score       : nan / 1.00
   Questions scored 1  : 0
   Questions scored 0  : 0

                                                       Question                                                                    Expected (short) Generated (short) Score
                       Comment s'est formé le système solaire ? Le système solaire s'est formé il y a environ 4,6 milliards d'années à partir d'...             ERROR  None
Qu'est-ce que la bioluminescence et comment les créatures de... La bioluminescence est la capacité de certains organismes à produire de la lumiè...             ERROR  None
Quelles créatures des abysses sont transparentes et pourquoi... Des créatures comme les méduses et certains calmars sont transparentes pour se c...             ERROR  None
                      How many exoplanets have been discovered? Thousands of exoplanets have been confirmed, with estimates suggesting billions ...             ERROR  None
P

In [1]:
import random

def shuffle():
    return random.choice(["JJK", "FMA"])

print(shuffle())

JJK
